# Routing BENCHMARK — Gemma 4 chess-coach (for the report)

Runs the **serious** routing ablation on Kaggle (T4, up to 12h — no Colab disconnect), over a large stratified sample of the validation set:

| condition | weights | base | isolates |
|---|---|---|---|
| **e4b-v4 adapter + harness** | trained v4 LoRA | E4B | the product |
| **e4b base + harness** | base (LoRA off) | E4B | what the **SFT weights** bought |
| **e2b adapter + harness** *(optional)* | prior E2B LoRA | E2B | the **prior production model** (E2B → E4B improvement) |

For each: confusion matrix + precision/recall/F1 + exact-name + format validity + throughput, then the deltas vs the product (adapter-vs-base = SFT weights on the same base; e4b-vs-e2b = the model-generation jump). The two E4B conditions reuse the SAME loaded model with the LoRA toggled (no 2nd load); the E2B condition has a different base, so it loads separately **after** the E4B model is freed (sequential — no OOM). No fabricated external baselines — reproducible from our own artifacts.

**The e2b condition needs its 5MB adapter** (`runs/gemma4_e2b_unified/best`, gitignored → not in the clone). Set `E2B_ADAPTER_DIR` (a Kaggle-Dataset upload) **or** `E2B_ADAPTER_REPO` (an HF push) in Cell 1; leave both blank to run the 2-condition E4B benchmark only.

**Setup:** Settings → Accelerator → **GPU T4 x2** (one is enough). Add **HF_TOKEN** under Add-ons → Secrets. Run top to bottom; the report + PNGs land in `/kaggle/working/` and render inline.

## CHAT-FOCUSED RUN — eyeball the model yourself

**Run these cells in order, skip the rest:**
1. **Cell 1** (config) — chat-focused defaults are already set.
2. **Cell 2** (deps + Stockfish) · 3. **Cell 3** (download base + adapter, ~9GB, the one wait).
4. **CHAT A/B** ← the centerpiece. Same hand-written prompts (slang/vague/tricky) under **board_on**
   (current live prompt) vs **board_off** (the clean trained prompt). Replies shown side by side, each
   web turn shows the **board BEFORE → AFTER** + a `board CHANGED` verdict, with **seconds + tok/s**.
   This is where you judge reply correctness and whether the live board injection hurts content.
5. *(optional)* **chat showcase** — the same chats once, rendered as slide PNG cards.
6. *(optional)* **completion eval (chess)** — runs at **parity (board hidden)**; gives the CORRECTED
   chess-completion number that replaces the bogus 13.5% artifact.
7. *(optional, last)* **cross-model line chart** — consolidates the measured numbers.

Everything else (the ~6h benchmark, version trend, native probe, confusion, GGUF/E2B) is OFF.
Rough time after download: CHAT A/B ~15 min · showcase ~7 min · chess completion ~10 min.
To go faster, set `RUN_CHAT_SHOWCASE=False` / `RUN_COMPLETION_CHESS=False` in Cell 1, or add `,thin`
to `CHAT_AB_VARIANTS` if you also want the rescue-layer-off variant.


In [ ]:
# Cell 1 - config
import os
BASE_REPO = 'unsloth/gemma-4-E4B-it'   # the base the adapter was trained on
REPO_URL  = 'https://github.com/RyanDev1st/llm-and-engine.git'
BRANCH    = 'feat/report-ppt-assets'
ADAPTER_REPO = 'RyanDev1st/gemma4-chesscoach-ckpt-v4'
TAG = 'best'
WORKDIR   = '/kaggle/working/llm-and-engine'
BASE_DIR  = '/kaggle/working/gemma4_e4b'
ADAPTER_DIR = '/kaggle/working/adapter/best'
PER_SLICE   = 25          # rows/slice (0 = full val). 25 is statistically strong.
TIME_BUDGET = 1800        # 30 min/condition CAP (fast final flight; was 3h for the full ablation)
# === FAST FINAL FLIGHT (~30-60 min) — measure the POST-FIX harness ====================================
# WHY these flags: the COMPLETION evals (Cell 6.7) + the TRANSCRIPT (Cell 5) run the REAL CoachLoop, so
# they REFLECT the harness fixes (verb-coercion / generic grounding / skill-body extract-first). The
# routing cells (E4B bench, version trend, native-mode) score RAW first-action and DON'T run the loop —
# the loop-level fixes are invisible to them, and they're already measured. So the fast flight runs the
# loop cells + keeps native ON (a fresh fair-test number for the report, capped at 30m/condition).
RUN_E4B_BENCH     = False  # Cell 4: the ~6h E4B val+stress routing benchmark (already captured)
RUN_VERSION_TREND = False  # Cell 6: per-version trend v2->v3->v4 (already measured; raw routing)
RUN_TRANSCRIPT    = False   # Cell 5: agent transcript — the ONLY proof the recipe-scaler over-ask fix landed (~3m)
RUN_NATIVE_PROBE  = False   # Cell 6.5: native-mode fair test (raw routing — won't move from loop fixes; capped ~30m)
RUN_COMPLETION    = False   # Cell 6.7a: COMPLETION eval on the OOD STRESS suite (engine-free) — generalization (~20m)
RUN_COMPLETION_CHESS = True    # Cell 6.7b: COMPLETION on CHESS val (needs Stockfish from Cell 2) — the product's chess half end to end
COMPLETION_CHESS_PER_SLICE = 4   # rows/slice for chess completion (bounded — engine analysis is slow even at a low timeout)
os.environ['CHESS_SF_TIMEOUT'] = '2.0'   # cap per-analysis engine wait for the bench (prod default 5.0; does NOT change depth/output)
# === e2b: DROPPED from scope (not shipping). Both blank => skipped entirely. ==========================
E2B_ADAPTER_DIR  = ''
E2B_ADAPTER_REPO = ''
E2B_BASE_REPO    = 'unsloth/gemma-4-E2B-it'
E2B_BASE_DIR     = '/kaggle/working/gemma4_e2b'
print(f'FAST FLIGHT | budget/cond {TIME_BUDGET//60}m | e4b_bench={RUN_E4B_BENCH} '
      f'version={RUN_VERSION_TREND} transcript={RUN_TRANSCRIPT} native={RUN_NATIVE_PROBE} '
      f'completion={RUN_COMPLETION} chess_completion={RUN_COMPLETION_CHESS} '
      f'| e2b={"on" if (E2B_ADAPTER_DIR or E2B_ADAPTER_REPO) else "off"}')

# --- GGUF quant A/B (HF nf4 vs Q5_K_M vs Q6_K) — quants must be pre-exported on Colab + uploaded ---
RUN_GGUF_AB  = False                  # Cell 6.8: completion A/B across backends (quality tradeoff)
GGUF_HF_REPO = ADAPTER_REPO           # HF repo holding the .gguf (default: the adapter repo)
GGUF_Q5_FILE = 'gemma4-E4B-chesscoach-Q5_K_M.gguf'   # '' to skip
GGUF_Q6_FILE = 'gemma4-E4B-chesscoach-Q6_K.gguf'     # '' to skip

# --- REPORT ASSETS (confusion matrix + authentic chats + cross-model line chart) -------------------
RUN_REPORT_GATE   = True   # GATE (CPU, ~5s): render EVERY report asset from seed data FIRST, so you
                           # can confirm the pipeline works and go to bed before the GPU cells finish.
RUN_CONFUSION     = False   # confusion matrix WITH its description baked into the PNG (tagged e4b-nf4)
RUN_CHAT_SHOWCASE = True    # the TWO authentic-chat sections (bare harness + web sandbox) + timing/tok-s
BUILD_MODEL_LINES = True    # assemble the cross-model line chart from whatever measured-*.json exist
ASSETS_DIR = f'{WORKDIR}/docs/findings/report_assets'

# --- REPLY-CORRECTNESS A/B (the FAST path: chats only, NO benchmark/eval) -----------------------
# Determine whether off-point answers come from the off-distribution LIVE BOARD prompt or the rescue
# layer, vs the weights. Same prompts under 3 serve configs, replies + board-before/after shown side
# by side. To run TONIGHT: run Cell 1, Cell 2 (deps), Cell 3 (download), then the CHAT A/B cell.
RUN_CHAT_AB = True
CHAT_AB_VARIANTS = 'board_on,board_off'   # drop any (e.g. 'board_on,board_off') to save time


In [ ]:
# Cell 2 - clone repo (prints HEAD so the commit is visible) + deps + Stockfish
import subprocess, sys, shutil
env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
if not os.path.exists(WORKDIR):
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,WORKDIR], check=True, env=env)
else:
    subprocess.run(['git','-C',WORKDIR,'fetch','origin',BRANCH], check=True, env=env)
    subprocess.run(['git','-C',WORKDIR,'reset','--hard',f'origin/{BRANCH}'], check=True, env=env)
print('HEAD:', subprocess.run(['git','-C',WORKDIR,'log','-1','--oneline'], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers','accelerate','peft','bitsandbytes','sentencepiece','protobuf','python-chess'], check=True)
os.environ['PYTHONPATH'] = f'{WORKDIR}/src/llm'
# Stockfish — needed by the CHESS completion eval (Cell 6.7b). backend.engine._resolve_sf() finds it
# on PATH or /usr/games/stockfish; we also export CHESS_SF so the subprocess resolves it deterministically.
sf = shutil.which('stockfish')
if not sf:
    subprocess.run(['bash','-lc','apt-get update -qq && apt-get install -y -qq stockfish'], check=False)
    sf = shutil.which('stockfish') or ('/usr/games/stockfish' if os.path.exists('/usr/games/stockfish') else None)
if sf:
    os.environ['CHESS_SF'] = sf
    print('stockfish:', sf)
else:
    print('stockfish: NOT FOUND -> chess completion (Cell 6.7b) will skip; OOD stress still runs')
import transformers, peft, bitsandbytes
print('transformers', transformers.__version__, '| peft', peft.__version__, '| bnb', bitsandbytes.__version__)

## GATE — run me FIRST (CPU, ~5s)
Renders every report asset from **seed data** so you can confirm the whole pipeline works, then trust the GPU cells and step away. No model, no GPU — just proves the renderers + chat suites are wired.


In [ ]:
# GATE - CPU smoke for the report assets (no model). Renders the confusion matrix (with its baked-in
# description), the cross-model line chart, and a chat card from SEED data; validates the hand-written
# chat suites. If it prints GATE OK + shows 3 SAMPLE images, every renderer works -- safe to launch the
# GPU cells below and go to bed.
import os, glob, subprocess, sys
env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
if not RUN_REPORT_GATE:
    print('RUN_REPORT_GATE=False -> skipping the CPU gate.')
else:
    subprocess.run([sys.executable, '-m', 'llm_training.report.gate', '--out', ASSETS_DIR],
                   cwd=f'{WORKDIR}/src/llm', env=env, check=True)
    from IPython.display import Image, display
    for png in sorted(glob.glob(f'{ASSETS_DIR}/SAMPLE-*.png')):
        print(png); display(Image(filename=png))
    print('\nGATE OK -- all report renderers produced an image. Safe to run the GPU cells and sleep.')


In [ ]:
# Cell 3 - download the E4B base (~9GB) + the v4 adapter (+ the tiny E2B adapter if its condition is on)
# NOTE: the E2B *base* (~10GB) is NOT downloaded here -- E4B+E2B bases together blow the ~20GB Kaggle
# disk. The last cell frees the E4B base first, then fetches the E2B base. Here we grab only the 5MB
# E2B adapter so it's ready.
from huggingface_hub import login, snapshot_download
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret('HF_TOKEN'))
except Exception:
    login(os.environ['HF_TOKEN'])
snapshot_download(repo_id=BASE_REPO, local_dir=BASE_DIR,
                  allow_patterns=['*.json','*.safetensors','*.model','*.txt','tokenizer*'])
snapshot_download(repo_id=ADAPTER_REPO, local_dir='/kaggle/working/adapter', allow_patterns=[f'{TAG}/*'])
need = ('adapter_model.safetensors', 'adapter_config.json')
assert all(os.path.exists(f'{ADAPTER_DIR}/{f}') for f in need), f'adapter not at {ADAPTER_DIR}'
print('e4b base + adapter ready:', os.listdir(ADAPTER_DIR))
E2B_ADAPTER = E2B_ADAPTER_DIR or None
if not E2B_ADAPTER and E2B_ADAPTER_REPO:          # the 5MB e2b adapter (NOT its base)
    snapshot_download(repo_id=E2B_ADAPTER_REPO, local_dir='/kaggle/working/e2b_adapter', allow_patterns=[f'{TAG}/*'])
    E2B_ADAPTER = f'/kaggle/working/e2b_adapter/{TAG}'
print('e2b adapter:', E2B_ADAPTER or 'OFF (skipping the e2b condition)')

In [ ]:
# === CHAT A/B - REPLY CORRECTNESS (run me after Cell 3) =========================================
# Same hand-written prompts under 3 serve configs so you can judge which gives APT content:
#   board_on  = current LIVE (LIVE BOARD line injected)   board_off = clean trained prompt
#   thin      = rescue/coverage layer dropped
# Each web turn shows the BOARD BEFORE -> AFTER + a 'board CHANGED' verdict, so you can SEE whether a
# move/new_game/load_fen tool actually took effect. Model loads ONCE; no benchmark, no eval. ~5-12 min.
import os, glob, shutil, subprocess, sys
os.environ['CHESS_HF_BASE'] = BASE_DIR
env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
if not RUN_CHAT_AB:
    print('RUN_CHAT_AB=False -> skipping.')
else:
    cmd = [sys.executable, '-m', 'llm_training.report.chat_ab', '--adapter', ADAPTER_DIR,
           '--variants', CHAT_AB_VARIANTS]
    print('==== running:', ' '.join(cmd), '====', flush=True)
    subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env=env, check=True)
    for f in glob.glob(f'{WORKDIR}/docs/findings/*-chat-ab.md'):
        shutil.copy(f, '/kaggle/working/')
    from IPython.display import Markdown, display
    md = sorted(glob.glob('/kaggle/working/*-chat-ab.md'))
    if md: display(Markdown(open(md[-1]).read()))


In [ ]:
# Cell 4 - RUN THE E4B BENCHMARK: 2 conditions x both tiers (hours-safe; writes reports + PNGs)
#  - val    = in-distribution held-out rows. Conditions: e4b-v4 adapter+harness, e4b base+harness.
#  - stress = held-out WILD rows: messy/slang/typo + UNSEEN out-of-domain catalogs + decline cases.
# The E4B model loads ONCE (adapter on = product, off = base). The E2B condition runs in the LAST
# cell. Set RUN_E4B_BENCH=False in Cell 1 to skip this ~6h step if you already have its report.
import os, sys, shutil, subprocess, glob
os.environ['CHESS_HF_BASE'] = BASE_DIR
if not RUN_E4B_BENCH:
    print('RUN_E4B_BENCH=False -> skipping the E4B benchmark (using your existing report).')
else:
    env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
    for suite, ps, tb in [('val', PER_SLICE, TIME_BUDGET), ('stress', 0, 0)]:
        cmd = [sys.executable, '-m', 'llm_training.eval_benchmark', '--adapter', ADAPTER_DIR,
               '--suite', suite, '--per-slice', str(ps), '--time-budget', str(tb)]
        print('\n==== running:', ' '.join(cmd), '====\n', flush=True)
        subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env=env, check=True)
    for f in glob.glob(f'{WORKDIR}/docs/findings/*-routing-benchmark*'):
        shutil.copy(f, '/kaggle/working/')
    from IPython.display import Image, Markdown, display
    for md in sorted(glob.glob('/kaggle/working/*-routing-benchmark*.md')):
        display(Markdown(open(md).read()))
    for png in sorted(glob.glob('/kaggle/working/*-routing-benchmark*.png')):
        print(png); display(Image(filename=png))

In [ ]:
# Cell 5 - CAPTURE A REAL AGENT TRANSCRIPT (for the report's "agent in action" section)
# Runs the FULL serve loop on held-out prompts over the real life-skills bundle (cooking/music/
# wellness/tax — absent from training): route by description -> load a real skill body -> call a
# real tool -> narrate the real result. Verbatim capture, not fabricated. Writes
# docs/findings/<date>-agent-transcript.md and shows it inline.
# Set RUN_TRANSCRIPT=False in Cell 1 to skip if you already captured it.
import os, sys, shutil, subprocess, glob
if not RUN_TRANSCRIPT:
    print('RUN_TRANSCRIPT=False -> skipping the transcript (using your existing capture).')
else:
    env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm', 'CHESS_HF_BASE': BASE_DIR}
    cmd = [sys.executable, '-m', 'llm_training.bench_transcript', '--adapter', ADAPTER_DIR]
    print('==== running:', ' '.join(cmd), '====\n', flush=True)
    subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env=env, check=True)
    for f in glob.glob(f'{WORKDIR}/docs/findings/*-agent-transcript.md'):
        shutil.copy(f, '/kaggle/working/')
    from IPython.display import Markdown, display
    md = sorted(glob.glob('/kaggle/working/*-agent-transcript.md'))
    if md: display(Markdown(open(md[-1]).read()))

In [ ]:
# Cell 6 - MEASURED per-version routing trend (v2 -> v3 -> v4) for the report's improvement chart
# Loops each trained adapter (downloaded fresh, freed between -> no OOM), runs the SAME fast routing
# eval, and writes: the timeline chart with REAL accuracy/version + per-slice bars + the v4 per-slice
# MISS analysis (settles slice G/H) + a trend report. A missing HF repo is SKIPPED, not fatal -- so
# if your v2/v3 ids differ, edit llm_training/report/chart_data.py VERSIONS (repo/sub) and re-run.
# Set RUN_VERSION_TREND=False in Cell 1 to skip (it's raw routing -> already measured, won't move).
import os, sys, shutil, subprocess, glob
os.environ['CHESS_HF_BASE'] = BASE_DIR
if not RUN_VERSION_TREND:
    print('RUN_VERSION_TREND=False -> skipping the version trend (using your existing measurement).')
else:
    env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
    cmd = [sys.executable, '-m', 'llm_training.report.version_eval',
           '--per-slice', '15', '--time-budget', '1800', '--workdir', '/kaggle/working/adapters']
    print('==== running:', ' '.join(cmd), '====\n', flush=True)
    subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env=env, check=True)
    for f in (glob.glob(f'{WORKDIR}/docs/findings/*-version-trend.md')
              + glob.glob(f'{WORKDIR}/docs/findings/report_assets/chart-*.png')
              + glob.glob(f'{WORKDIR}/docs/findings/report_assets/*-misses.jsonl')):
        shutil.copy(f, '/kaggle/working/')
    from IPython.display import Image, Markdown, display
    md = sorted(glob.glob('/kaggle/working/*-version-trend.md'))
    if md: display(Markdown(open(md[-1]).read()))
    for png in sorted(glob.glob('/kaggle/working/chart-*.png')):
        print(png); display(Image(filename=png))

In [ ]:
# Cell 6.5 - NATIVE-MODE PROOF: adapter vs base on the mode-dependent slices (the FAIR test)
# The main benchmark (Cell 4) forces FAST mode for speed -- cheap, equal handicap to both conditions,
# but it collapses the trained goal/think->action sequence on AUTO/THINK-trained slices (that's why
# slice G read 0%: the model emitted the goal keyword <skill>threats</skill> instead of chess-coach).
# Here we score each row in its TRAINED reasoning mode (--native-mode, bigger token budget) on the
# slices that depend on it. This is the honest apples-to-apples comparison and it's where the trained
# adapter pulls DECISIVELY ahead of the base on the same harness. Writes a *-val-native report.
import os, sys, shutil, subprocess, glob
os.environ['CHESS_HF_BASE'] = BASE_DIR
if not RUN_NATIVE_PROBE:
    print('RUN_NATIVE_PROBE=False -> skipping the native-mode proof.')
else:
    env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
    cmd = [sys.executable, '-m', 'llm_training.eval_benchmark', '--adapter', ADAPTER_DIR,
           '--suite', 'val', '--native-mode', '--max-new-tokens', '200', '--per-slice', '25',
           '--time-budget', str(TIME_BUDGET),
           '--slices', 'G,H,E,F,V1_S_compound_plan,V1_T_audited_plan,V1_R_compute_grounding']
    print('==== running:', ' '.join(cmd), '====\n', flush=True)
    subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env=env, check=True)
    for f in glob.glob(f'{WORKDIR}/docs/findings/*-routing-benchmark-val-native*'):
        shutil.copy(f, '/kaggle/working/')
    from IPython.display import Image, Markdown, display
    for md in sorted(glob.glob('/kaggle/working/*-routing-benchmark-val-native.md')):
        display(Markdown(open(md).read()))
    for png in sorted(glob.glob('/kaggle/working/*-routing-benchmark-val-native*.png')):
        print(png); display(Image(filename=png))

In [ ]:
# Cell 6.7 - COMPLETION EVAL: the FULL serve loop, not just first-action routing (the metric the peer
# reviews said was missing). Two runs share the loaded E4B+v4:
#   (a) OOD STRESS (life-skills, engine-free)  -> generalization to UNSEEN domains
#   (b) CHESS VAL (needs Stockfish from Cell 2) -> the product's CHESS half, end to end
# Each scores completed / exec_ok / args_ok / grounded + `recovered` (a wrong first route the loop self-
# corrected to a grounded answer) AND prints a FAILING-ROWS table so a low metric is explained, not
# guessed. Runs BEFORE any base-freeing. Set RUN_COMPLETION / RUN_COMPLETION_CHESS in Cell 1.
import os, sys, shutil, subprocess
os.environ['CHESS_HF_BASE'] = BASE_DIR
# TRAIN/SERVE PARITY: 94% of chess val rows expect board_state in the chain, and the model trained
# with NO board line in the prompt. The completion LOOP injects the LIVE BOARD (build_system_prompt),
# which makes board_state redundant -> the model skips it -> 'completed' fails despite correct
# routing (the 13.5% artifact). Eval at parity (board hidden) so the trained tool chain fires.
os.environ['CHESS_BOARD_HOOK'] = '0'
env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
if RUN_COMPLETION:
    cmd = [sys.executable, '-m', 'llm_training.eval_completion', '--adapter', ADAPTER_DIR,
           '--stress', '--time-budget', '1800', '--tag', 'e4b-nf4']
    print('\n==== completion [OOD STRESS]:', ' '.join(cmd), '====\n', flush=True)
    subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env=env, check=True)
if RUN_COMPLETION_CHESS:
    sf = shutil.which('stockfish') or ('/usr/games/stockfish' if os.path.exists('/usr/games/stockfish') else None)
    if not sf:
        print('\nRUN_COMPLETION_CHESS=True but Stockfish not found -> skipping the chess completion run.')
    else:
        cmd = [sys.executable, '-m', 'llm_training.eval_completion', '--adapter', ADAPTER_DIR,
               '--per-slice', str(COMPLETION_CHESS_PER_SLICE), '--time-budget', '1800']
        print(f'\n==== completion [CHESS VAL] sf={sf} sf_timeout={os.environ.get("CHESS_SF_TIMEOUT")}:',
              ' '.join(cmd), '====\n', flush=True)
        subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env=env, check=True)

In [ ]:
# REPORT - CONFUSION MATRIX with its description baked INTO the png (one copy-paste for a slide).
# Tagged e4b-nf4 so its verb accuracy feeds the cross-model line chart. ~10-20 min at per-slice 25.
import os, glob, shutil, subprocess, sys
os.environ['CHESS_HF_BASE'] = BASE_DIR
env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
if not RUN_CONFUSION:
    print('RUN_CONFUSION=False -> skipping.')
else:
    cmd = [sys.executable, '-m', 'llm_training.eval_confusion', '--adapter', ADAPTER_DIR,
           '--per-slice', '25', '--time-budget', '1200', '--tag', 'e4b-nf4']
    print('==== running:', ' '.join(cmd), '====', flush=True)
    subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env=env, check=True)
    for f in glob.glob(f'{WORKDIR}/docs/findings/*-routing-confusion.*'):
        shutil.copy(f, '/kaggle/working/')
    from IPython.display import Image, Markdown, display
    for mdf in sorted(glob.glob('/kaggle/working/*-routing-confusion.md')):
        display(Markdown(open(mdf).read()))
    for png in sorted(glob.glob('/kaggle/working/*-routing-confusion.png')):
        print(png); display(Image(filename=png))


In [ ]:
# REPORT - AUTHENTIC CHATS: 2 sections (bare harness, then the chess-web sandbox). Real model, real
# harness, hand-written realistic prompts (slang/vague/tricky); EVERY turn prints time(s) + tokens +
# tok/s. Tagged e4b-nf4 so the mean tok/s feeds the cross-model chart. ~5-10 min.
import os, glob, shutil, subprocess, sys
os.environ['CHESS_HF_BASE'] = BASE_DIR
env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
if not RUN_CHAT_SHOWCASE:
    print('RUN_CHAT_SHOWCASE=False -> skipping.')
else:
    cmd = [sys.executable, '-m', 'llm_training.report.chat_showcase', '--adapter', ADAPTER_DIR,
           '--tag', 'e4b-nf4']
    print('==== running:', ' '.join(cmd), '====', flush=True)
    subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env=env, check=True)
    for f in glob.glob(f'{WORKDIR}/docs/findings/*-chat-showcase.md'):
        shutil.copy(f, '/kaggle/working/')
    from IPython.display import Image, Markdown, display
    mdf = sorted(glob.glob('/kaggle/working/*-chat-showcase.md'))
    if mdf: display(Markdown(open(mdf[-1]).read()))
    for png in sorted(glob.glob(f'{ASSETS_DIR}/chat-section*.png')):
        print(png); display(Image(filename=png))


In [ ]:
# REPORT - GGUF QUANT A/B (optional). For each pre-exported quant (Q5_K_M, Q6_K) download it from HF,
# run the SAME completion + confusion evals tagged e4b-q5 / e4b-q6 (so the quants land on the line
# chart), then DELETE it to respect Kaggle disk. Needs the .gguf uploaded to GGUF_HF_REPO (export on
# Colab). Gated by RUN_GGUF_AB (Cell 1, default off). Best-effort: a missing file just skips that quant.
import os, glob, subprocess, sys
env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
if not RUN_GGUF_AB:
    print('RUN_GGUF_AB=False -> skipping the GGUF quant A/B (line chart shows the quants as n/a).')
else:
    from huggingface_hub import hf_hub_download
    for tag, fname in [('e4b-q5', GGUF_Q5_FILE), ('e4b-q6', GGUF_Q6_FILE)]:
        if not fname:
            print(f'{tag}: no file set -> skip'); continue
        try:
            gguf = hf_hub_download(repo_id=GGUF_HF_REPO, filename=fname, local_dir='/kaggle/working/gguf')
        except Exception as exc:
            print(f'{tag}: cannot download {fname} from {GGUF_HF_REPO}: {exc} -> skip'); continue
        for mod, extra in [('llm_training.eval_completion', ['--stress', '--time-budget', '1200']),
                           ('llm_training.eval_confusion', ['--per-slice', '15', '--time-budget', '900'])]:
            cmd = [sys.executable, '-m', mod, '--gguf', gguf, '--tag', tag] + extra
            print('==== running:', ' '.join(cmd), '====', flush=True)
            subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env=env, check=False)
        try: os.remove(gguf)
        except OSError: pass
        print(f'{tag}: done + freed {fname}')


In [ ]:
# Cell 7 - PRIOR E2B PRODUCTION MODEL (run LAST -- it frees the E4B base from disk)
# E4B base (~9GB) + E2B base (~10GB) can't both fit Kaggle's ~20GB disk. So this runs AFTER all the
# E4B work (Cells 4-6): eval_benchmark --e2b-only deletes the E4B base (--free-base), downloads the
# E2B base, then evaluates the prior E2B adapter on the SAME val rows -> a routing report tagged 'e2b'
# (verb acc / exact-name / confusion / per-slice), directly comparable to the E4B numbers above.
import os, sys, shutil, subprocess, glob
if not E2B_ADAPTER:
    print('E2B condition OFF -- set E2B_ADAPTER_REPO or E2B_ADAPTER_DIR in Cell 1 to run it.')
else:
    env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
    cmd = [sys.executable, '-m', 'llm_training.eval_benchmark', '--adapter', ADAPTER_DIR,
           '--suite', 'val', '--per-slice', str(PER_SLICE), '--time-budget', str(TIME_BUDGET),
           '--e2b-only', '--e2b-adapter', E2B_ADAPTER, '--e2b-base', E2B_BASE_DIR,
           '--e2b-base-repo', E2B_BASE_REPO, '--free-base', BASE_DIR]
    print('==== running:', ' '.join(cmd), '====\n', flush=True)
    subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env=env, check=True)
    for f in glob.glob(f'{WORKDIR}/docs/findings/*-routing-benchmark-e2b*'):
        shutil.copy(f, '/kaggle/working/')
    from IPython.display import Image, Markdown, display
    for md in sorted(glob.glob('/kaggle/working/*-routing-benchmark-e2b.md')):
        display(Markdown(open(md).read()))
    for png in sorted(glob.glob('/kaggle/working/*-routing-benchmark-e2b*.png')):
        print(png); display(Image(filename=png))

In [ ]:
# REPORT - CROSS-MODEL LINE CHART (run LAST). Collects every measured-*.json the cells above wrote
# (verb from the confusion cell, completion/grounded from Cell 6.7, tok/s from the chats, the GGUF
# quants from the A/B if you ran it) onto the model seeds and renders the line across ALL models:
# E2B prior -> E4B base -> E4B nf4 (current) -> E4B Q5_K_M -> E4B Q6_K. A model with no measured
# number is drawn as a GAP, never faked.
import os, subprocess, sys
env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
if not BUILD_MODEL_LINES:
    print('BUILD_MODEL_LINES=False -> skipping.')
else:
    pycode = ('from pathlib import Path;'
              'from llm_training.report import chart_data as D, ppt_charts, measured;'
              'm=measured.collect(r"%s");'
              'models=D.merge_measured(D.MODELS, m);'
              'ppt_charts.model_lines(models, Path(r"%s")/"chart-model-lines.png");'
              'print(D.model_table_md(models));'
              'print("measured tags:", sorted(m))') % (ASSETS_DIR, ASSETS_DIR)
    subprocess.run([sys.executable, '-c', pycode], cwd=f'{WORKDIR}/src/llm', env=env, check=True)
    from IPython.display import Image, display
    p = f'{ASSETS_DIR}/chart-model-lines.png'
    if os.path.exists(p): print(p); display(Image(filename=p))
